In [1]:
import os, json, math
import numpy as np
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:

# ============================================================
# 1) Data I/O (your text_min format)
# ============================================================
def load_textmin_dataset(dataset_folder: str):
    edges_path = os.path.join(dataset_folder, "edges.txt")
    feat_path = os.path.join(dataset_folder, "features.csv")
    lab_path = os.path.join(dataset_folder, "labels.csv")

    if not os.path.exists(edges_path): raise FileNotFoundError(edges_path)
    if not os.path.exists(feat_path): raise FileNotFoundError(feat_path)
    if not os.path.exists(lab_path): raise FileNotFoundError(lab_path)

    edges = np.loadtxt(edges_path, dtype=np.int64)
    X = np.loadtxt(feat_path, delimiter=",", dtype=np.float32)
    y = np.loadtxt(lab_path, dtype=np.int64)

    if edges.ndim == 1:
        edges = edges.reshape(-1, 2)

    N = X.shape[0]
    if y.shape[0] != N:
        raise ValueError(f"labels.csv length {y.shape[0]} != num_nodes {N}")

    row, col = edges[:, 0], edges[:, 1]
    A = sp.coo_matrix((np.ones(len(row), dtype=np.float32), (row, col)), shape=(N, N))
    A = A + A.T
    A.data[:] = 1.0
    A.setdiag(0)
    A.eliminate_zeros()
    return X, y.astype(np.int64), A.tocsr()


def _read_idx_file(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing split file: {path}")
    idx = np.loadtxt(path, dtype=np.int64)
    return np.atleast_1d(idx)


def load_20_splits(splits_dir: str, dataset: str, one_based_idx: bool):
    splits = []
    for s in range(1, 21):
        seed_dir = os.path.join(splits_dir, dataset, f"seed_{s:02d}")
        tr = _read_idx_file(os.path.join(seed_dir, "train_idx.txt"))
        va = _read_idx_file(os.path.join(seed_dir, "val_idx.txt"))
        te = _read_idx_file(os.path.join(seed_dir, "test_idx.txt"))
        if one_based_idx:
            tr = tr - 1
            va = va - 1
            te = te - 1
        splits.append({"seed": s, "train": tr, "val": va, "test": te})
    return splits


def detect_one_based(tr, va, te, N: int) -> bool:
    mx = int(max(tr.max(), va.max(), te.max()))
    mn = int(min(tr.min(), va.min(), te.min()))
    if mx == N and mn >= 1:
        return True
    return False



In [3]:
def tfidf_l2_svd_train_only(
X: np.ndarray,
train_idx: np.ndarray,
pca_dim: int = 256,
use_tfidf: bool = True,
l2_rows: bool = True,
eps: float = 1e-12,
) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    train_idx = np.asarray(train_idx, dtype=np.int64)
    n_train = int(len(train_idx))
    if n_train < 2:
        raise ValueError("Need at least 2 training nodes for PCA/SVD(train-only).")
    Xw = X
    if use_tfidf:
        df = (X[train_idx] > 0).sum(axis=0).astype(np.float32)
        idf = np.log((n_train + 1.0) / (df + 1.0)) + 1.0
        Xw = X * idf[None, :]
    if l2_rows:
        norms = np.linalg.norm(Xw, axis=1, keepdims=True)
        Xw = Xw / (norms + eps)
    rnk = int(max(1, min(pca_dim, Xw.shape[1], n_train - 1)))
    mu = Xw[train_idx].mean(axis=0, keepdims=True)
    Xc_train = Xw[train_idx] - mu
    try:
        from sklearn.utils.extmath import randomized_svd
        U, S, Vt = randomized_svd(Xc_train, n_components=rnk, n_iter=3, random_state=0)
    except Exception:
        U, S, Vt = np.linalg.svd(Xc_train, full_matrices=False)
    W = Vt[:rnk].T # [F, rnk]
    Xp = (Xw - mu) @ W
    return Xp.astype(np.float32)

In [4]:
def normalize_gcn_adj(A: sp.spmatrix, add_self_loops: bool = True, eps: float = 1e-12):
    A = A.tocsr()
    if add_self_loops:
        A = A + sp.eye(A.shape[0], format="csr", dtype=np.float32)
    deg = np.asarray(A.sum(axis=1)).reshape(-1).astype(np.float32)
    deg_inv_sqrt = 1.0 / np.sqrt(deg + eps)
    D_inv_sqrt = sp.diags(deg_inv_sqrt)
    A_hat = (D_inv_sqrt @ A @ D_inv_sqrt).tocsr()
    return A_hat


def sparse_to_torch_coo(A: sp.spmatrix):
    A = A.tocoo()
    idx = torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)
    val = torch.tensor(A.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(idx, val, size=A.shape).coalesce()


def spmm(A_coo: torch.Tensor, X: torch.Tensor):
    return torch.sparse.mm(A_coo, X)


In [8]:
class GCNWeakLearner(nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int, dropout: float):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden, bias=True)
        self.lin2 = nn.Linear(hidden, out_dim, bias=True)
        self.dropout = float(dropout)
        
    def forward(self, X, A_hat):
        H = spmm(A_hat, X)
        H = self.lin1(H)
        H = F.relu(H)
        H = F.dropout(H, p=self.dropout, training=self.training)
        H = spmm(A_hat, H)
        out = self.lin2(H)
        return out


@torch.no_grad()
def accuracy(logits: torch.Tensor, y: torch.Tensor, idx: torch.Tensor) -> float:
    pred = logits.argmax(dim=-1)
    return float((pred[idx] == y[idx]).float().mean().item())


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def weighted_ce_loss(logits: torch.Tensor, y: torch.Tensor, tr_idx: torch.Tensor, w: torch.Tensor):
    logp = F.log_softmax(logits[tr_idx], dim=-1) # [n_tr, C]
    yi = y[tr_idx]
    nll = -logp[torch.arange(tr_idx.numel(), device=logp.device), yi] # [n_tr]
    wi = w[tr_idx] # [n_tr]
    return (wi * nll).sum() / (wi.sum() + 1e-12)

def train_weak_learner(
model: nn.Module,
X: torch.Tensor,
y: torch.Tensor,
A_hat: torch.Tensor,
tr: torch.Tensor,
va: torch.Tensor,
w: torch.Tensor,
lr: float,
wd: float,
max_epochs: int,
patience: int,
):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val = -1.0
    best_state = None
    bad = 0
    for _ in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        out = model(X, A_hat)
        loss = weighted_ce_loss(out, y, tr, w)
        loss.backward()
        opt.step()
        model.eval()
        out = model(X, A_hat)
        val_acc = accuracy(out, y, va)
        if val_acc > best_val + 1e-6:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return float(best_val)

def run_one_split_adagcn(
X_np: np.ndarray,
y_np: np.ndarray,
A: sp.spmatrix,
train_idx: np.ndarray,
val_idx: np.ndarray,
test_idx: np.ndarray,
device: str = "cuda",
hidden: int = 64,
dropout: float = 0.5,
lr: float = 0.01,
wd: float = 5e-4,
T: int = 8, # number of boosting stages
stage_epochs: int = 200, # max epochs per weak learner
stage_patience: int = 30, # early stopping per weak learner
seed: int = 1,
):
    set_seed(seed)
    dev = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")
    N = X_np.shape[0]
    n_classes = int(np.max(y_np) + 1)
    A_hat = normalize_gcn_adj(A, add_self_loops=True)
    A_hat = sparse_to_torch_coo(A_hat).to(dev)
    X = torch.tensor(X_np, dtype=torch.float32, device=dev)
    y = torch.tensor(y_np, dtype=torch.long, device=dev)
    tr = torch.tensor(train_idx, dtype=torch.long, device=dev)
    va = torch.tensor(val_idx, dtype=torch.long, device=dev)
    te = torch.tensor(test_idx, dtype=torch.long, device=dev)
    w = torch.zeros(N, device=dev, dtype=torch.float32)
    w[tr] = 1.0 / float(tr.numel())
    F_ens = torch.zeros((N, n_classes), device=dev, dtype=torch.float32)
    alphas = []
    eps = 1e-6
    for t in range(1, T + 1):
        model = GCNWeakLearner(in_dim=X.shape[1], hidden=hidden, out_dim=n_classes, dropout=dropout).to(dev)
        best_val_stage = train_weak_learner(
        model=model, X=X, y=y, A_hat=A_hat,
        tr=tr, va=va, w=w,
        lr=lr, wd=wd,
        max_epochs=stage_epochs, patience=stage_patience
        )
        model.eval()
        logits = model(X, A_hat) # [N,C]
        pred_tr = logits[tr].argmax(dim=-1)
        wrong = (pred_tr != y[tr]).float()
        err = (w[tr] * wrong).sum() / (w[tr].sum() + 1e-12)
        err = float(err.item())
        err = min(max(err, eps), 1.0 - eps)
        alpha = 0.5 * math.log((1.0 - err) / err)
        alphas.append(float(alpha))
        F_ens = F_ens + float(alpha) * logits
        w_tr_new = w[tr] * torch.exp(torch.tensor(alpha, device=dev) * wrong)
        w.zero_()
        w[tr] = w_tr_new / (w_tr_new.sum() + 1e-12)
        tr_acc = accuracy(F_ens, y, tr)
        va_acc = accuracy(F_ens, y, va)
        te_acc = accuracy(F_ens, y, te)
        print(f"[stage {t:02d}/{T}] err={err:.4f} alpha={alpha:.4f} "
        f"| ens train={tr_acc:.4f} val={va_acc:.4f} test={te_acc:.4f} "
        f"| best_val_stage={best_val_stage:.4f}")
    tr_acc = accuracy(F_ens, y, tr)
    va_acc = accuracy(F_ens, y, va)
    te_acc = accuracy(F_ens, y, te)
    return {
        "train_acc": float(tr_acc),
        "val_acc": float(va_acc),
        "test_acc": float(te_acc),
        "test_err": float(1.0 - te_acc),
        "T": int(T),
        "alphas": [float(a) for a in alphas],
    }


def run_20_splits_adagcn(
data_dir: str, # contains <dataset>/{edges.txt,features.csv,labels.csv}
splits_dir: str, # contains <dataset>/seed_01/{train,val,test}_idx.txt
dataset: str,
out_dir: str,
one_based_idx: bool | None = None, # True if txt are 1..N, False if 0..N-1, None=auto
# preprocessing (per seed)
use_prep: bool = True,
pca_dim: int = 256,
use_tfidf: bool = True,
l2_rows: bool = True,
# training
device: str = "cuda",
hidden: int = 64,
dropout: float = 0.5,
lr: float = 0.01,
wd: float = 5e-4,
# boosting
T: int = 8,
stage_epochs: int = 200,
stage_patience: int = 30,
):
    X_raw, y, A = load_textmin_dataset(os.path.join(data_dir, dataset))
    N = X_raw.shape[0]
    splits = load_20_splits(splits_dir, dataset, one_based_idx=False) # load raw first
    if one_based_idx is None:
        tr0, va0, te0 = splits[0]["train"], splits[0]["val"], splits[0]["test"]
        one_based_idx = detect_one_based(tr0, va0, te0, N)
        print(f"[auto] one_based_idx = {one_based_idx}")
    if one_based_idx:
        for sp_ in splits:
            sp_["train"] = sp_["train"] - 1
            sp_["val"] = sp_["val"] - 1
            sp_["test"] = sp_["test"] - 1
    results = []
    for sp_ in splits:
        seed = int(sp_["seed"])
        tr = sp_["train"].astype(np.int64)
        va = sp_["val"].astype(np.int64)
        te = sp_["test"].astype(np.int64)
        mx = max(int(tr.max()), int(va.max()), int(te.max()))
        mn = min(int(tr.min()), int(va.min()), int(te.min()))
        if mx >= N or mn < 0:
            raise IndexError(
            f"Split indices out of range for N={N}. Min={mn}, Max={mx}. "
            f"Try one_based_idx=True or one_based_idx=None(auto)."
            )
        if use_prep:
            pca_dim_run = int(max(1, min(pca_dim, len(tr) - 1, X_raw.shape[1])))
            Xp = tfidf_l2_svd_train_only(
            X_raw, tr,
            pca_dim=pca_dim_run,
            use_tfidf=use_tfidf,
            l2_rows=l2_rows
            )
        else:
            Xp = X_raw.astype(np.float32)
        print("\n" + "=" * 70)
        print(f"[seed {seed:02d}] AdaGCN | N={N} F={Xp.shape[1]} K={int(y.max()+1)} "
        f"| train={len(tr)} val={len(va)} test={len(te)}")
        print("=" * 70)
        out = run_one_split_adagcn(
            X_np=Xp, y_np=y, A=A,
            train_idx=tr, val_idx=va, test_idx=te,
            device=device,
            hidden=hidden,
            dropout=dropout,
            lr=lr, wd=wd,
            T=T,
            stage_epochs=stage_epochs,
            stage_patience=stage_patience,
            seed=seed
        )
        out["seed"] = seed
        results.append(out)
        print(f"[seed {seed:02d}] FINAL test_acc={out['test_acc']:.4f} test_err={out['test_err']:.4f}")
    test_acc = np.array([r["test_acc"] for r in results], dtype=np.float32)
    test_err = np.array([r["test_err"] for r in results], dtype=np.float32)
    summary = {
    "dataset": dataset,
    "model": "AdaGCN (AdaBoost + 2-layer GCN weak learners, PyTorch)",
    "splits": {"one_based_idx": bool(one_based_idx)},
    "use_prep": bool(use_prep),
    "prep": {
    "use_tfidf": bool(use_tfidf),
    "l2_rows": bool(l2_rows),
    "pca_dim": int(pca_dim),
    "train_only": True
    } if use_prep else None,
        "training": {
            "device": device,
            "hidden": int(hidden),
            "dropout": float(dropout),
            "lr": float(lr),
            "weight_decay": float(wd),
            "T": int(T),
            "stage_epochs": int(stage_epochs),
            "stage_patience": int(stage_patience),
    },
    "mean_test_acc": float(test_acc.mean()),
    "std_test_acc": float(test_acc.std(ddof=1)),
    "mean_test_err": float(test_err.mean()),
    "std_test_err": float(test_err.std(ddof=1)),
    "per_seed": results,
    }
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{dataset}_AdaGCN_20splits.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print("\n=== FINAL SUMMARY (AdaGCN) ===")
    print(f"mean_test_acc = {summary['mean_test_acc']:.4f} ± {summary['std_test_acc']:.4f}")
    print(f"mean_test_err = {summary['mean_test_err']:.4f} ± {summary['std_test_err']:.4f}")
    print(f"Saved JSON: {out_path}")
    return summary



In [6]:
def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--splits_dir", required=True)
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--out_dir", default="results_adagcn")
    ap.add_argument("--one_based_idx", action="store_true", help="If split files are 1..N (will subtract 1).")
    ap.add_argument("--auto_idx", action="store_true", help="Auto-detect index base (recommended).")
    ap.add_argument("--use_prep", action="store_true")
    ap.add_argument("--pca_dim", type=int, default=256)
    ap.add_argument("--no_tfidf", action="store_true")
    ap.add_argument("--no_l2", action="store_true")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--hidden", type=int, default=64)
    ap.add_argument("--dropout", type=float, default=0.5)
    ap.add_argument("--lr", type=float, default=0.01)
    ap.add_argument("--wd", type=float, default=5e-4)
    ap.add_argument("--T", type=int, default=8)
    ap.add_argument("--stage_epochs", type=int, default=200)
    ap.add_argument("--stage_patience", type=int, default=30)
    args = ap.parse_args()
    if args.auto_idx:
        one_based = None
    else:
        one_based = bool(args.one_based_idx)
    run_20_splits_adagcn(
        data_dir=args.data_dir,
        splits_dir=args.splits_dir,
        dataset=args.dataset,
        out_dir=args.out_dir,
        one_based_idx=one_based,
        use_prep=bool(args.use_prep),
        pca_dim=int(args.pca_dim),
        use_tfidf=not bool(args.no_tfidf),
        l2_rows=not bool(args.no_l2),
        device=args.device,
        hidden=int(args.hidden),
        dropout=float(args.dropout),
        lr=float(args.lr),
        wd=float(args.wd),
        T=int(args.T),
        stage_epochs=int(args.stage_epochs),
        stage_patience=int(args.stage_patience),
    )

In [10]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\citation_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\citeseer_e2eBoost",
    dataset = "citeseer",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]

[auto] one_based_idx = True

[seed 01] AdaGCN | N=4230 F=256 K=6 | train=2535 val=844 test=851
[stage 01/8] err=0.0276 alpha=1.7807 | ens train=0.9724 val=0.9467 test=0.9495 | best_val_stage=0.9467
[stage 02/8] err=0.0393 alpha=1.5982 | ens train=0.9803 val=0.9491 test=0.9495 | best_val_stage=0.9562
[stage 03/8] err=0.0268 alpha=1.7953 | ens train=0.9874 val=0.9479 test=0.9471 | best_val_stage=0.9396
[stage 04/8] err=0.0587 alpha=1.3871 | ens train=0.9913 val=0.9491 test=0.9471 | best_val_stage=0.9384
[stage 05/8] err=0.0449 alpha=1.5290 | ens train=0.9953 val=0.9479 test=0.9483 | best_val_stage=0.9336
[stage 06/8] err=0.0254 alpha=1.8228 | ens train=0.9976 val=0.9479 test=0.9459 | best_val_stage=0.9372
[stage 07/8] err=0.0692 alpha=1.2995 | ens train=0.9976 val=0.9455 test=0.9471 | best_val_stage=0.9289
[stage 08/8] err=0.0739 alpha=1.2643 | ens train=0.9980 val=0.9431 test=0.9471 | best_val_stage=0.9360
[seed 01] FINAL test_acc=0.9471 test_err=0.0529

[seed 02] AdaGCN | N=4230 F=256 

(0.9470623731613159, 0.00850343331694603)

In [11]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\wekb_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\Cornell_e2eBoost",
    dataset = "Cornell",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]

[auto] one_based_idx = True

[seed 01] AdaGCN | N=183 F=107 K=5 | train=108 val=35 test=40
[stage 01/8] err=0.4444 alpha=0.1116 | ens train=0.5556 val=0.4857 test=0.4000 | best_val_stage=0.4857
[stage 02/8] err=0.5679 alpha=-0.1366 | ens train=0.2037 val=0.2000 test=0.2000 | best_val_stage=0.4571
[stage 03/8] err=0.5341 alpha=-0.0683 | ens train=0.2037 val=0.2000 test=0.2000 | best_val_stage=0.4571
[stage 04/8] err=0.5121 alpha=-0.0242 | ens train=0.2037 val=0.2000 test=0.2250 | best_val_stage=0.4571
[stage 05/8] err=0.5128 alpha=-0.0255 | ens train=0.2037 val=0.2000 test=0.2250 | best_val_stage=0.4571
[stage 06/8] err=0.4984 alpha=0.0032 | ens train=0.2037 val=0.2000 test=0.2250 | best_val_stage=0.4571
[stage 07/8] err=0.4606 alpha=0.0790 | ens train=0.2037 val=0.2000 test=0.2000 | best_val_stage=0.4857
[stage 08/8] err=0.5447 alpha=-0.0895 | ens train=0.2037 val=0.2000 test=0.2250 | best_val_stage=0.4571
[seed 01] FINAL test_acc=0.2250 test_err=0.7750

[seed 02] AdaGCN | N=183 F=107 

(0.38875001668930054, 0.16730742156505585)

In [14]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\wekb_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\Texas_e2eBoost",
    dataset = "Texas",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]

[auto] one_based_idx = True

[seed 01] AdaGCN | N=183 F=121 K=5 | train=122 val=40 test=46
[stage 01/8] err=0.2951 alpha=0.4354 | ens train=0.7049 val=0.6000 test=0.5652 | best_val_stage=0.6000
[stage 02/8] err=0.4230 alpha=0.1552 | ens train=0.6885 val=0.5750 test=0.5652 | best_val_stage=0.5750
[stage 03/8] err=0.4984 alpha=0.0032 | ens train=0.6885 val=0.5750 test=0.5652 | best_val_stage=0.5750
[stage 04/8] err=0.4514 alpha=0.0974 | ens train=0.7049 val=0.5750 test=0.5870 | best_val_stage=0.5750
[stage 05/8] err=0.5207 alpha=-0.0414 | ens train=0.7049 val=0.5750 test=0.5870 | best_val_stage=0.5750
[stage 06/8] err=0.4584 alpha=0.0834 | ens train=0.7049 val=0.5750 test=0.5870 | best_val_stage=0.5750
[stage 07/8] err=0.5380 alpha=-0.0762 | ens train=0.7131 val=0.5750 test=0.5870 | best_val_stage=0.5750
[stage 08/8] err=0.4745 alpha=0.0510 | ens train=0.7131 val=0.5750 test=0.5870 | best_val_stage=0.5750
[seed 01] FINAL test_acc=0.5870 test_err=0.4130

[seed 02] AdaGCN | N=183 F=121 K=5

(0.561956524848938, 0.045858751982450485)

In [12]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\wekb_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\Winsconsin_e2eBoost",
    dataset = "Winsconsin",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]

[auto] one_based_idx = True

[seed 01] AdaGCN | N=251 F=148 K=5 | train=149 val=49 test=53
[stage 01/8] err=0.4362 alpha=0.1282 | ens train=0.5638 val=0.5510 test=0.4717 | best_val_stage=0.5510
[stage 02/8] err=0.3873 alpha=0.2293 | ens train=0.6644 val=0.5510 test=0.4717 | best_val_stage=0.5714
[stage 03/8] err=0.4045 alpha=0.1933 | ens train=0.6711 val=0.5510 test=0.4717 | best_val_stage=0.5714
[stage 04/8] err=0.5597 alpha=-0.1200 | ens train=0.6711 val=0.5510 test=0.4717 | best_val_stage=0.5918
[stage 05/8] err=0.1936 alpha=0.7133 | ens train=0.8121 val=0.5102 test=0.4151 | best_val_stage=0.5510
[stage 06/8] err=0.3448 alpha=0.3209 | ens train=0.8322 val=0.5102 test=0.4528 | best_val_stage=0.5510
[stage 07/8] err=0.5521 alpha=-0.1046 | ens train=0.8322 val=0.5102 test=0.4528 | best_val_stage=0.5306
[stage 08/8] err=0.5362 alpha=-0.0724 | ens train=0.8322 val=0.5102 test=0.4528 | best_val_stage=0.5306
[seed 01] FINAL test_acc=0.4528 test_err=0.5472

[seed 02] AdaGCN | N=251 F=148 K=

(0.5245283246040344, 0.05427171662449837)

In [ ]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\wiki_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\chameleon_e2eBoost2",
    dataset = "chameleon",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]

[auto] one_based_idx = True

[seed 01] AdaGCN | N=2277 F=256 K=5 | train=1364 val=454 test=459
[stage 01/8] err=0.2852 alpha=0.4594 | ens train=0.7148 val=0.6167 test=0.5861 | best_val_stage=0.6167
[stage 02/8] err=0.3503 alpha=0.3090 | ens train=0.7177 val=0.6035 test=0.5969 | best_val_stage=0.6211
[stage 03/8] err=0.3642 alpha=0.2785 | ens train=0.7221 val=0.6057 test=0.5969 | best_val_stage=0.6123
[stage 04/8] err=0.4103 alpha=0.1814 | ens train=0.7258 val=0.6079 test=0.6013 | best_val_stage=0.6189
[stage 05/8] err=0.4222 alpha=0.1568 | ens train=0.7287 val=0.6123 test=0.5991 | best_val_stage=0.6101
[stage 06/8] err=0.4527 alpha=0.0950 | ens train=0.7339 val=0.6145 test=0.5991 | best_val_stage=0.6079
[stage 07/8] err=0.4697 alpha=0.0607 | ens train=0.7339 val=0.6167 test=0.6013 | best_val_stage=0.6189
[stage 08/8] err=0.5037 alpha=-0.0073 | ens train=0.7346 val=0.6167 test=0.6013 | best_val_stage=0.6057
[seed 01] FINAL test_acc=0.6013 test_err=0.3987

[seed 02] AdaGCN | N=2277 F=256

In [ ]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\wiki_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\squirrel_e2eBoost2",
    dataset = "squirrel",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]

In [ ]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\actor_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\Actor_e2eBoost",
    dataset = "Actor",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]

In [ ]:
summary = run_20_splits_adagcn(
    data_dir = r"C:\Users\user\Desktop\Thesis Coding\wikics_text_min",
    splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\WikiCS_e2eBoost2",
    dataset = "WikiCS",
    out_dir = r"C:\Users\user\Desktop\Thesis Coding\results_adagcn",
    
    one_based_idx=None, 
    use_prep=True,
    pca_dim=256,
    device="cuda",
    T=8,
    stage_epochs=200,
    stage_patience=30
)

summary["mean_test_acc"], summary["std_test_acc"]